In [11]:
import pandas as pd

ipl_url = "https://raw.githubusercontent.com/kkamal11/EDA_on_IPL/main/IPL_Matches_2008_2022.csv"
ipl_df = pd.read_csv(ipl_url)

print(ipl_df.head())

        ID       City        Date Season  MatchNumber  \
0  1312200  Ahmedabad  2022-05-29   2022        Final   
1  1312199  Ahmedabad  2022-05-27   2022  Qualifier 2   
2  1312198    Kolkata  2022-05-25   2022   Eliminator   
3  1312197    Kolkata  2022-05-24   2022  Qualifier 1   
4  1304116     Mumbai  2022-05-22   2022           70   

                         Team1                 Team2  \
0             Rajasthan Royals        Gujarat Titans   
1  Royal Challengers Bangalore      Rajasthan Royals   
2  Royal Challengers Bangalore  Lucknow Super Giants   
3             Rajasthan Royals        Gujarat Titans   
4          Sunrisers Hyderabad          Punjab Kings   

                              Venue            TossWinner TossDecision  \
0  Narendra Modi Stadium, Ahmedabad      Rajasthan Royals          bat   
1  Narendra Modi Stadium, Ahmedabad      Rajasthan Royals        field   
2             Eden Gardens, Kolkata  Lucknow Super Giants        field   
3             Eden Garde

In [12]:
print("Rows before cleaning:", len(ipl_df))

ipl_df = ipl_df.dropna(subset=["WinningTeam"]).reset_index(drop=True)

print("Rows after cleaning:", len(ipl_df))
print(ipl_df[["WinningTeam"]].head())

Rows before cleaning: 950
Rows after cleaning: 946
                   WinningTeam
0               Gujarat Titans
1             Rajasthan Royals
2  Royal Challengers Bangalore
3               Gujarat Titans
4                 Punjab Kings


In [13]:
import numpy as np

margin = pd.to_numeric(ipl_df["Margin"], errors="coerce").fillna(0)
won_by = ipl_df["WonBy"].fillna("").str.lower()

win_by_runs = margin.where(won_by == "runs", 0)
win_by_wickets = margin.where(won_by == "wickets", 0)

ipl_df["match_margin_type"] = np.select(
    [win_by_runs > 0, win_by_wickets > 0],
    ["Runs", "Wickets"],
    default="Tie/No Result"
)

print(ipl_df[["WonBy", "Margin", "match_margin_type"]].head(10))
print(ipl_df["match_margin_type"].value_counts())

     WonBy  Margin match_margin_type
0  Wickets     7.0           Wickets
1  Wickets     7.0           Wickets
2     Runs    14.0              Runs
3  Wickets     7.0           Wickets
4  Wickets     5.0           Wickets
5  Wickets     5.0           Wickets
6  Wickets     5.0           Wickets
7  Wickets     8.0           Wickets
8     Runs     2.0              Runs
9     Runs     3.0              Runs
match_margin_type
Wickets          509
Runs             423
Tie/No Result     14
Name: count, dtype: int64


In [14]:
team_cities = pd.DataFrame({
    "team": [
        "Mumbai Indians", "Chennai Super Kings",
        "Kolkata Knight Riders", "Royal Challengers Bangalore",
        "Rajasthan Royals", "Sunrisers Hyderabad",
        "Delhi Capitals", "Punjab Kings",
        "Gujarat Titans", "Lucknow Super Giants"
    ],
    "home_city": [
        "Mumbai", "Chennai", "Kolkata", "Bengaluru",
        "Jaipur", "Hyderabad", "Delhi", "Mohali",
        "Ahmedabad", "Lucknow"
    ]
})

ipl_df = ipl_df.merge(
    team_cities.rename(columns={"team": "Team1", "home_city": "team1_home_city"}),
    on="Team1",
    how="left"
).merge(
    team_cities.rename(columns={"team": "Team2", "home_city": "team2_home_city"}),
    on="Team2",
    how="left"
)

print(ipl_df[["Team1", "team1_home_city", "Team2", "team2_home_city"]].head(10))

                         Team1 team1_home_city                        Team2  \
0             Rajasthan Royals          Jaipur               Gujarat Titans   
1  Royal Challengers Bangalore       Bengaluru             Rajasthan Royals   
2  Royal Challengers Bangalore       Bengaluru         Lucknow Super Giants   
3             Rajasthan Royals          Jaipur               Gujarat Titans   
4          Sunrisers Hyderabad       Hyderabad                 Punjab Kings   
5               Delhi Capitals           Delhi               Mumbai Indians   
6          Chennai Super Kings         Chennai             Rajasthan Royals   
7               Gujarat Titans       Ahmedabad  Royal Challengers Bangalore   
8         Lucknow Super Giants         Lucknow        Kolkata Knight Riders   
9          Sunrisers Hyderabad       Hyderabad               Mumbai Indians   

  team2_home_city  
0       Ahmedabad  
1          Jaipur  
2         Lucknow  
3       Ahmedabad  
4          Mohali  
5         

In [15]:
wins_by_season = ipl_df.pivot_table(
    index="Season",
    columns="WinningTeam",
    values="ID",
    aggfunc="count",
    fill_value=0
)

display(wins_by_season)

selected_season = ipl_df["Season"].iloc[0]
print(f"\nTop 3 teams by wins in {selected_season}:")
print(wins_by_season.loc[selected_season].sort_values(ascending=False).head(3))S

WinningTeam,Chennai Super Kings,Deccan Chargers,Delhi Capitals,Delhi Daredevils,Gujarat Lions,Gujarat Titans,Kings XI Punjab,Kochi Tuskers Kerala,Kolkata Knight Riders,Lucknow Super Giants,Mumbai Indians,Pune Warriors,Punjab Kings,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Sunrisers Hyderabad
Season,,,,,,,,,,,,,,,,,,
2007/08,9,2,0,7,0,0,10,0,6,0,7,0,0,13,0,0,4,0
2009,8,9,0,10,0,0,7,0,3,0,5,0,0,6,0,0,9,0
2009/10,9,8,0,7,0,0,4,0,7,0,11,0,0,6,0,0,8,0
2011,11,6,0,4,0,0,7,6,8,0,10,4,0,6,0,0,10,0
2012,10,4,0,11,0,0,8,0,12,0,10,4,0,7,0,0,8,0
2013,12,0,0,3,0,0,8,0,6,0,13,4,0,11,0,0,9,10
2014,10,0,0,2,0,0,12,0,11,0,7,0,0,7,0,0,5,6
2015,10,0,0,5,0,0,3,0,7,0,10,0,0,7,0,0,8,7
2016,0,0,0,7,9,0,4,0,8,0,7,0,0,0,0,5,9,11



Top 3 teams by wins in 2022:
WinningTeam
Gujarat Titans                 12
Rajasthan Royals               10
Royal Challengers Bangalore     9
Name: 2022, dtype: int64
